In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
start_year = 2000
end_year = 2024
pagination_limit = 1000  # Pagination limit for requests


indicators = {
    "population": "SP_POP_TOTL",
    "land_area_km2": "AG_LND_TOTL_K2",
    "gdp_per_capita_ppp_current": "NY_GDP_PCAP_PP_CD",
    "gdp_per_capita_usd_current": "NY_GDP_PCAP_CD",
    "life_expectancy": "SP_DYN_LE00_IN",
    "co2_emissions_per_capita": "EN_GHG_CO2_PC_CE_AR5",
    "pm2_5_exposure": "EN_ATM_PM25_MC_M3",
    'urban_population_percentage': "SP_URB_TOTL_IN_ZS",
    'health_expenditure_percentage_of_gdp': "SH_XPD_CHEX_GD_ZS",
}

# Indicator 1 : Population

## 1. Test code

In [3]:
df_test = pd.DataFrame({'REF_AREA': ['FRA', 'FRA', 'FRA','FRA','ITA'],
                       'TIME_PERIOD': [2020, 2020, 2021, 2021, 2021],
                       'OBS_VALUE': [100, 200, 200, np.nan, 400]})

# Key: country x date, value: OBS_VALUE
# Expected: 3 groups: (FRA, 2020), (FRA, 2021) and (ITA, 2021)
grouped = df_test.groupby(['REF_AREA', 'TIME_PERIOD']).size()
#print(grouped)

# 1. Filter the Series
filtered = grouped[grouped > 1]

print(f"Filtered: {filtered}")


# 2. Get the number of elements
filtered_count = filtered.count()  # or len(filtered)

# 3. Get the keys (the index)
keys = filtered.index.tolist()

#print(f"Matching keys: {keys}")

Filtered: REF_AREA  TIME_PERIOD
FRA       2020           2
          2021           2
dtype: int64


## 2. Row uniqueness

Check that each key (year × area) has exactly one row.

In [4]:
df_pop = pd.read_csv(Path(f"../data/raw/population_{start_year}-{end_year}_raw.csv"))
print("================================")
print(f"indicator: ", "population")
print(f"File read for indicator population: ",df_pop.shape)

# Group by key
pop_counts_per_key = df_pop.groupby(['TIME_PERIOD', 'REF_AREA']).size()

# 1. Filter the Series
df_pop_filtered = pop_counts_per_key[pop_counts_per_key > 1]

# 2. Get the number of elements
df_pop_count = df_pop_filtered.count()  # or len(filtered)

# 3. Get the keys (the index)
df_pop_keys = df_pop_filtered.index.tolist()

print(f"Elements > 1: {df_pop_count}")
print(f"Matching keys: {df_pop_keys}")


indicator:  population
File read for indicator population:  (6600, 24)
Elements > 1: 0
Matching keys: []


Check for missing values in OBS_VALUE and whether the number of keys equals the number of rows.

In [5]:
# Number of years
df_pop_years = df_pop['TIME_PERIOD'].nunique()
print(f"Number of years: {df_pop_years}")

# Number of countries
df_pop_countries = df_pop['REF_AREA'].nunique()
print(f"Number of countries: {df_pop_countries}")

nb_rows = df_pop.shape[0]
nb_keys = df_pop['REF_AREA'].nunique() * df_pop['TIME_PERIOD'].nunique()
print(f"Number of keys: {nb_keys}")
print(f"Number of rows: {nb_rows}")

nb_missing_values = df_pop['OBS_VALUE'].isnull().sum()
print(f"Number of missing values: {nb_missing_values}")


Number of years: 25
Number of countries: 264
Number of keys: 6600
Number of rows: 6600
Number of missing values: 0


Everything looks correct.

Check the number of areas: 264 is far above the expected number of countries.

In [6]:
#df_test = df_pop

a = df_test.nunique()
print(a)
a['OBS_VALUE']

set_a = set(a)
print(set(a))

REF_AREA       2
TIME_PERIOD    2
OBS_VALUE      3
dtype: int64
{2, 3}


In [7]:
df_pop = pd.read_csv(Path(f"../data/raw/population_{start_year}-{end_year}_raw.csv"))


#we drop all columns with a unique value count of 1 or 0 except units
df_pop = df_pop[['REF_AREA', 'TIME_PERIOD', 'OBS_VALUE', 'UNIT_MULT', 'UNIT_MEASURE','UNIT_TYPE']]
print(df_pop) 

# we drop units if unnecessary
df_pop = df_pop[['REF_AREA', 'TIME_PERIOD', 'OBS_VALUE']]
print(df_pop)

     REF_AREA  TIME_PERIOD   OBS_VALUE  UNIT_MULT UNIT_MEASURE  UNIT_TYPE
0         YEM         2015    31159379          0           PS        NaN
1         ZMB         2015    16399089          0           PS        NaN
2         ZWE         2015    14399013          0           PS        NaN
3         AFE         2016   623369401          0           PS        NaN
4         AFW         2016   428797338          0           PS        NaN
...       ...          ...         ...        ...          ...        ...
6595      SSF         2015  1024659421          0           PS        NaN
6596      SSA         2015  1024566002          0           PS        NaN
6597      TSS         2015  1024659421          0           PS        NaN
6598      UMC         2015  2916662642          0           PS        NaN
6599      WLD         2015  7441084771          0           PS        NaN

[6600 rows x 6 columns]
     REF_AREA  TIME_PERIOD   OBS_VALUE
0         YEM         2015    31159379
1        

In [8]:
print(df_pop['REF_AREA'].unique())

<StringArray>
['YEM', 'ZMB', 'ZWE', 'AFE', 'AFW', 'ARB', 'CSS', 'CEB', 'EAR', 'EAS',
 ...
 'SLB', 'SOM', 'ZAF', 'SSD', 'DMA', 'SAS', 'TSA', 'SSF', 'DOM', 'ECU']
Length: 264, dtype: str


In [9]:
print(df_pop['REF_AREA'].nunique())

264


There are 264 areas, which include countries and **aggregates**.

Drop the aggregates and keep countries only.

In [10]:
import requests

per_page = 400 # Large enough to fetch everything in one call (total < per_page)


url = "https://api.worldbank.org/v2/country?"
response = requests.get(url,params={"format":"json", "per_page": per_page})
response.raise_for_status()

print(response.url)
print(response.text[:300])

area_data = response.json()


print("Number of country IDs",area_data[0]['total'])
print("country_id_aruba",area_data[1][0]['id'])
print("country_id_2",area_data[1][1]['id'])

print("nb countries",len(area_data[1]))

L_id_countries = []

for country_index in range(len(area_data[1])):
    L_id_countries.append(area_data[1][country_index]['id'])
print(L_id_countries)
print(len(L_id_countries))

https://api.worldbank.org/v2/country?format=json&per_page=400
[{"page":1,"pages":1,"per_page":"400","total":296},[{"id":"ABW","iso2Code":"AW","name":"Aruba","region":{"id":"LCN","iso2code":"ZJ","value":"Latin America & Caribbean "},"adminregion":{"id":"","iso2code":"","value":""},"incomeLevel":{"id":"HIC","iso2code":"XD","value":"High income"},"lendingType":{"
Number of country IDs 296
country_id_aruba ABW
country_id_2 AFE
nb countries 296
['ABW', 'AFE', 'AFG', 'AFR', 'AFW', 'AGO', 'ALB', 'AND', 'ARB', 'ARE', 'ARG', 'ARM', 'ASM', 'ATG', 'AUS', 'AUT', 'AZE', 'BDI', 'BEA', 'BEC', 'BEL', 'BEN', 'BFA', 'BGD', 'BGR', 'BHI', 'BHR', 'BHS', 'BIH', 'BLA', 'BLR', 'BLZ', 'BMN', 'BMU', 'BOL', 'BRA', 'BRB', 'BRN', 'BSS', 'BTN', 'BWA', 'CAA', 'CAF', 'CAN', 'CEA', 'CEB', 'CEU', 'CHE', 'CHI', 'CHL', 'CHN', 'CIV', 'CLA', 'CME', 'CMR', 'COD', 'COG', 'COL', 'COM', 'CPV', 'CRI', 'CSA', 'CSS', 'CUB', 'CUW', 'CYM', 'CYP', 'CZE', 'DEA', 'DEC', 'DEU', 'DJI', 'DLA', 'DMA', 'DMN', 'DNK', 'DNS', 'DOM', 'DSA', 'D

In [11]:
import pandas as pd

df_area = pd.json_normalize(area_data[1])
# Should have 295  rows
df_area.shape

(296, 18)

In [12]:
df_area.head()

,id,iso2Code,name,capitalCity,longitude,latitude,region.id,region.iso2code,region.value,adminregion.id,adminregion.iso2code,adminregion.value,incomeLevel.id,incomeLevel.iso2code,incomeLevel.value,lendingType.id,lendingType.iso2code,lendingType.value
0,ABW,AW,Aruba,Oranjestad,-70.0167,12.5167,LCN,ZJ,Latin America & Caribbean,,,,HIC,XD,High income,LNX,XX,Not classified
1,AFE,ZH,Africa Eastern and Southern,,,,NA,NA,Aggregates,,,,NA,NA,Aggregates,,,Aggregates
2,AFG,AF,Afghanistan,Kabul,69.1761,34.5228,MEA,ZQ,"Middle East, North Africa, Afghanistan & Pakistan",MNA,XQ,"Middle East, North Africa, Afghanistan & Pakis...",LIC,XM,Low income,IDX,XI,IDA
3,AFR,A9,Africa,,,,NA,NA,Aggregates,,,,NA,NA,Aggregates,,,Aggregates
4,AFW,ZI,Africa Western and Central,,,,NA,NA,Aggregates,,,,NA,NA,Aggregates,,,Aggregates


In [13]:
df_area['region.value'].value_counts()

df_area[df_area['region.value']!= 'Aggregates'].shape

(217, 18)

### Data Cleaning 

In [14]:
df_pop = pd.read_csv(Path(f"../data/raw/population_{start_year}-{end_year}_raw.csv"))
df_pop = df_pop[['REF_AREA', 'TIME_PERIOD', 'OBS_VALUE']]

print(df_pop.shape)
df_pop.head()

(6600, 3)


,REF_AREA,TIME_PERIOD,OBS_VALUE
0,YEM,2015,31159379
1,ZMB,2015,16399089
2,ZWE,2015,14399013
3,AFE,2016,623369401
4,AFW,2016,428797338


In [15]:
df_area = pd.read_csv(Path('../data/external/area_data.csv'))
print("with aggregates",df_area.shape)

# Drop aggregates
df_countries = df_area[df_area['region.value']!= 'Aggregates']
df_countries.head()
print("Number of aggregates in df_area :" ,df_area.shape[0] - df_countries.shape[0])

# Drop every ID that is not in df_countries
df_pop_filtered = df_pop[df_pop['REF_AREA'].isin(df_countries['id'])]
df_pop_filtered.head()
print(df_pop_filtered['REF_AREA'].nunique())

# Did we drop too many?
df_pop_dropped = df_pop[~df_pop['REF_AREA'].isin(df_countries['id'])]
print('number of IDs dropped from df_pop :', df_pop_dropped['REF_AREA'].nunique())

# Are all dropped IDs aggregates?
df_aggregates_from_area = df_area[df_area['region.value'] == 'Aggregates']
df_dropped_countries= df_pop_dropped[~df_pop_dropped['REF_AREA'].isin(df_aggregates_from_area['id'])]

print('number of countries dropped :',df_dropped_countries['REF_AREA'].nunique())

#print('number of IDs dropped from df_pop :', df_pop.shape[0]-df_pop_filtered.shape[0])
#print("inverse mask", ~df_pop['REF_AREA'].isin(df_countries['id']))
#print("mask", df_pop['REF_AREA'].isin(df_countries['id']))

with aggregates (295, 18)
Number of aggregates in df_area : 78
217
number of IDs dropped from df_pop : 47
number of countries dropped : 0


## Loop over all indicators to check columns

In [16]:
start_year = 2000
end_year = 2024
pagination_limit = 1000  # Pagination limit for requests


indicators = {
    "population": "SP_POP_TOTL",
    "land_area_km2": "AG_LND_TOTL_K2",
    "gdp_per_capita_ppp_current": "NY_GDP_PCAP_PP_CD",
    "gdp_per_capita_usd_current": "NY_GDP_PCAP_CD",
    "life_expectancy": "SP_DYN_LE00_IN",
    "co2_emissions_per_capita": "EN_GHG_CO2_PC_CE_AR5",
    "pm2_5_exposure": "EN_ATM_PM25_MC_M3",
    'urban_population_percentage': "SP_URB_TOTL_IN_ZS",
    'health_expenditure_percentage_of_gdp': "SH_XPD_CHEX_GD_ZS",
}

KNOWN_EXTRA_AGGREGATES = ['FCS'] # FCS = Fragile and Conflict-Affected Situations

In [17]:
number_of_countries_expected = 217

df_area = pd.read_csv(Path('../data/external/area_data.csv'))
df_countries = df_area[df_area['region.value']!= 'Aggregates']

df_countries_id = df_countries[['id']]
df_years= pd.DataFrame([k for k in range(start_year,end_year+1)],dtype=np.dtype("int64"))
#print(df_countries_id)
#print(df_years)                        
df_all_indicators = df_countries_id.merge(df_years, how='cross')
print(df_all_indicators)
df_all_indicators.shape
df_all_indicators.columns
df_all_indicators = df_all_indicators.rename(columns={'id' : 'REF_AREA', 0 : 'TIME_PERIOD',})

       id     0
0     ABW  2000
1     ABW  2001
2     ABW  2002
3     ABW  2003
4     ABW  2004
...   ...   ...
5420  ZWE  2020
5421  ZWE  2021
5422  ZWE  2022
5423  ZWE  2023
5424  ZWE  2024

[5425 rows x 2 columns]


In [18]:
number_of_countries_expected = 217

df_area = pd.read_csv(Path('../data/external/area_data.csv'))
df_countries = df_area[df_area['region.value']!= 'Aggregates']



for name, code in indicators.items():
    print("==================")
    print("indicator:", name)
    df = pd.read_csv(Path(f"../data/raw/{name}_{start_year}-{end_year}_raw.csv"))
    # Units checks

    #print("unit multiplier:",df['UNIT_MULT'])
    # Drop unused columns
    df = df[['REF_AREA', 'TIME_PERIOD', 'OBS_VALUE', 'UNIT_MEASURE']]

    # Check for duplicate keys
    counts_per_key = df.groupby(['TIME_PERIOD', 'REF_AREA']).size()
    df_duplicates = counts_per_key[counts_per_key > 1]
    if df_duplicates.count() != 0:
        raise ValueError(f"Duplicate keys found for {name}")

    # Dropping aggregates and checking if the 217 countries are here
    
    # Drop every ID that is not in df_countries
    df_filtered = df[df['REF_AREA'].isin(df_countries['id'])] # IDs conserved
    #if df_filtered['REF_AREA'].nunique() != number_of_countries_expected:
       # raise ValueError(f"The number of countries is wrong : {df_filtered['REF_AREA'].nunique()}. We expected {number_of_countries_expected}")
    
    # Are all dropped IDs aggregates?
    df_dropped = df[~df['REF_AREA'].isin(df_countries['id'])] # IDs dropped
    df_aggregates_from_area = df_area[df_area['region.value'] == 'Aggregates']
    df_dropped_countries= df_dropped[
        ~df_dropped['REF_AREA'].isin(df_aggregates_from_area['id'])
        &
        ~df_dropped['REF_AREA'].isin(KNOWN_EXTRA_AGGREGATES)
        ]
    if df_dropped_countries['REF_AREA'].nunique() != 0:
        raise ValueError(f"Countries were dropped, not only aggregates")




    # Check that all years and areas are present
    nb_years = df_filtered['TIME_PERIOD'].nunique()
    #print(f"Number of years: {nb_years}")

    nb_countries = df_filtered['REF_AREA'].nunique()
    #print(f"Number of countries: {nb_countries}")

    nb_rows = df_filtered.shape[0]
    nb_keys = df_filtered['REF_AREA'].nunique() * df_filtered['TIME_PERIOD'].nunique()
    #print(f"Number of keys: {nb_keys}")
    #print(f"Number of rows: {nb_rows}")

    #print("file coverage: ", nb_rows/5425)

    # Check for missing values
    nb_missing_values = df_filtered['OBS_VALUE'].isnull().sum()
    #print(f"Number of missing values: {nb_missing_values}")



    ################################################################
    ### Merging ###
    ################################################################

    #Renaming columns
    df_to_merge = df_filtered.rename(columns={'OBS_VALUE' : f'{name}_{df_filtered['UNIT_MEASURE'].iloc[0]}'})
    #df_to_merge = df_to_merge.drop(['UNIT_MEASURE', 'LATEST_DATA'], axis=1)
    df_to_merge = df_to_merge.drop(['UNIT_MEASURE'], axis=1)
    
  
    #print(df_to_merge.head())

    shape_0_before_merge = df_all_indicators.shape[0]
    df_all_indicators = df_all_indicators.merge(df_to_merge,how='left', on=['REF_AREA', 'TIME_PERIOD'])
    if df_all_indicators.shape[0] !=shape_0_before_merge :
        raise ValueError(f"merge error : the number of row has changed")


indicator: population


indicator: land_area_km2
indicator: gdp_per_capita_ppp_current
indicator: gdp_per_capita_usd_current
indicator: life_expectancy
indicator: co2_emissions_per_capita
indicator: pm2_5_exposure
indicator: urban_population_percentage
indicator: health_expenditure_percentage_of_gdp


In [19]:
df_all_indicators.shape

(5425, 11)

In [20]:

# Lowercase column names
df_all_indicators.columns = [x.lower() for x in df_all_indicators.columns]

# Removing a "km_2" because of csv name : "land_area_km2_2000-2024_raw"
df_all_indicators = df_all_indicators.rename(columns={'land_area_km2_km2' : 'land_area_km2'})
df_all_indicators.head()

,ref_area,time_period,population_ps,land_area_km2,gdp_per_capita_ppp_current_ppp,gdp_per_capita_usd_current_usd,life_expectancy_yr,co2_emissions_per_capita_t,pm2_5_exposure_mcg_m3,urban_population_percentage_pt,health_expenditure_percentage_of_gdp_pt_gdp
0,ABW,2000,90588,180.0,30245.706974,20681.023027,72.939,2.968384,NaN,65.354550,NaN
1,ABW,2001,91439,180.0,31920.239073,20740.132583,73.044,2.973567,NaN,65.335114,NaN
2,ABW,2002,92074,180.0,31888.508737,21307.248251,73.135,3.225666,NaN,65.282069,NaN
3,ABW,2003,93128,180.0,32507.084315,21949.485996,73.236,3.676660,NaN,65.198292,NaN
4,ABW,2004,95138,180.0,35059.273098,23700.631990,73.223,3.672560,NaN,65.088653,NaN
